# Ad targeting — offline evaluation of a candidate bidding policy

Mirror of `examples/use_cases/02_ad_targeting.py`. Use this pattern on
any counterfactual-ads log (e.g., Criteo) by replacing `make_synth_logs`
with a public dataset loader once issue #70 ships.

👉 [Open in Colab](https://colab.research.google.com/github/dgenio/skdr-eval/blob/main/examples/notebooks/04_ad_targeting.ipynb)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet skdr-eval scikit-learn

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

import skdr_eval

logs, creatives, _ = skdr_eval.make_synth_logs(n=2000, n_ops=4, seed=11)
skdr_eval.validate_logs(logs)
list(creatives)

In [ ]:
models = {
    "HGB_bidder": HistGradientBoostingRegressor(max_iter=80, random_state=11),
    "Ridge_bidder": Ridge(alpha=1.0, random_state=11),
}
artifact = skdr_eval.evaluate_sklearn_models(
    logs=logs,
    models=models,
    fit_models=True,
    n_splits=2,
    outcome_estimator="hgb",
    random_state=11,
    policy_train="pre_split",
    policy_train_frac=0.8,
    clip_grid=(2.0, 5.0, 10.0, 20.0, 50.0, float("inf")),
)
cols = [
    c
    for c in ("model", "estimator", "V_hat", "ESS", "support_health", "pareto_k")
    if c in artifact.report.columns
]
artifact.report[cols].round(4)

`pareto_k ≥ 0.7` is a red flag — the weight distribution has heavy
tails and DR variance estimates may be unreliable. SNDR is more
robust in this regime; large DR / SNDR disagreement is itself a
trust signal.